In [1]:
# imports and load filtered data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# load full dataset
df = pd.read_csv('../data/PS_20174392719_1491204439457_log.csv')

# apply the filter we discovered in EDA — only TRANSFER and CASH_OUT
df = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

print(f"Shape: {df.shape}")
print("Loaded and filtered ✓")

Shape: (2770409, 11)
Loaded and filtered ✓


In [2]:
# Cell 2 — Feature 1: mule account flag
df['is_mule_account'] = (df['oldbalanceDest'] == 0).astype(int)

print("is_mule_account distribution:")
print(df['is_mule_account'].value_counts())
print(f"\nFraud rate when is_mule_account=1: {df[df['is_mule_account']==1]['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate when is_mule_account=0: {df[df['is_mule_account']==0]['isFraud'].mean()*100:.2f}%")

is_mule_account distribution:
is_mule_account
0    2381089
1     389320
Name: count, dtype: int64

Fraud rate when is_mule_account=1: 1.37%
Fraud rate when is_mule_account=0: 0.12%


In [3]:
# Feature 2: amount to balance ratio
# avoid division by zero by adding a small value
df['amount_to_balance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)

print("Amount to balance ratio:")
print(f"\nLegitimate mean: {df[df['isFraud']==0]['amount_to_balance_ratio'].mean():.4f}")
print(f"Fraudulent mean: {df[df['isFraud']==1]['amount_to_balance_ratio'].mean():.4f}")

Amount to balance ratio:

Legitimate mean: 158167.3983
Fraudulent mean: 1161.9667


In [4]:
# clip the ratio to handle extreme values
df['amount_to_balance_ratio'] = df['amount_to_balance_ratio'].clip(upper=10)

print("After clipping at 10:")
print(f"\nLegitimate mean: {df[df['isFraud']==0]['amount_to_balance_ratio'].mean():.4f}")
print(f"Fraudulent mean: {df[df['isFraud']==1]['amount_to_balance_ratio'].mean():.4f}")

After clipping at 10:

Legitimate mean: 7.7502
Fraudulent mean: 1.0206


In [5]:
# Cell 5 — Feature 3: sender account drained
df['sender_account_drained'] = (df['newbalanceOrig'] == 0).astype(int)

print("sender_account_drained distribution:")
print(df['sender_account_drained'].value_counts())
print(f"\nFraud rate when drained=1: {df[df['sender_account_drained']==1]['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate when drained=0: {df[df['sender_account_drained']==0]['isFraud'].mean()*100:.2f}%")

sender_account_drained distribution:
sender_account_drained
1    2496656
0     273753
Name: count, dtype: int64

Fraud rate when drained=1: 0.32%
Fraud rate when drained=0: 0.06%


In [6]:
# Feature 4: destination balance unchanged after receiving
df['dest_balance_unchanged'] = (
    df['oldbalanceDest'] == df['newbalanceDest']
).astype(int)

print("dest_balance_unchanged distribution:")
print(df['dest_balance_unchanged'].value_counts())
print(f"\nFraud rate when unchanged=1: {df[df['dest_balance_unchanged']==1]['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate when unchanged=0: {df[df['dest_balance_unchanged']==0]['isFraud'].mean()*100:.2f}%")

dest_balance_unchanged distribution:
dest_balance_unchanged
0    2764617
1       5792
Name: count, dtype: int64

Fraud rate when unchanged=1: 70.55%
Fraud rate when unchanged=0: 0.15%


In [7]:
# Feature 5: encode transaction type
df['is_transfer'] = (df['type'] == 'TRANSFER').astype(int)

print("is_transfer distribution:")
print(df['is_transfer'].value_counts())
print(f"\nFraud rate for TRANSFER: {df[df['is_transfer']==1]['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate for CASH_OUT: {df[df['is_transfer']==0]['isFraud'].mean()*100:.2f}%")

is_transfer distribution:
is_transfer
0    2237500
1     532909
Name: count, dtype: int64

Fraud rate for TRANSFER: 0.77%
Fraud rate for CASH_OUT: 0.18%


In [8]:
# Cell 8 — finalize dataset for modeling
df_model = df[[
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest',
    'is_transfer',
    'is_mule_account',
    'amount_to_balance_ratio',
    'sender_account_drained',
    'dest_balance_unchanged',
    'isFraud'
]].copy()

print(f"Final dataset shape: {df_model.shape}")
print(f"\nColumns: {df_model.columns.tolist()}")
print(f"\nFraud cases: {df_model['isFraud'].sum():,}")
print(f"Legitimate cases: {(df_model['isFraud']==0).sum():,}")
print("\nFirst 5 rows:")
df_model.head()

Final dataset shape: (2770409, 11)

Columns: ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'is_transfer', 'is_mule_account', 'amount_to_balance_ratio', 'sender_account_drained', 'dest_balance_unchanged', 'isFraud']

Fraud cases: 8,213
Legitimate cases: 2,762,196

First 5 rows:


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,is_transfer,is_mule_account,amount_to_balance_ratio,sender_account_drained,dest_balance_unchanged,isFraud
2,181.00,181.0,0.0,0.0,0.00,1,1,0.994505,1,1,1
3,181.00,181.0,0.0,21182.0,0.00,0,0,0.994505,1,0,1
15,229133.94,15325.0,0.0,5083.0,51513.44,0,0,10.000000,1,0,0
19,215310.30,705.0,0.0,22425.0,0.00,1,0,10.000000,1,0,0
24,311685.89,10835.0,0.0,6267.0,2719172.89,1,0,10.000000,1,0,0


In [10]:
#  save processed dataset
df_model.to_csv('../data/processed_data.csv', index=False)
print("Saved to data/processed_data.csv ✓")
print(f"File size: {df_model.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

Saved to data/processed_data.csv ✓
File size: 253.6 MB
